Define the source and target portals:

In [1]:
from arcgis.gis import *
source = GIS("https://portalhostds.ags.esri.com/gis", "admin", "esri.agp2")
target = GIS("https://ps001563.esri.com/portal", "portaladmin", "esri.agp")

#Users
List the users in the source and target portals:

In [2]:
sourceusers = source.users.search()
sourceusers

[<User username:admin>,
 <User username:automate1>,
 <User username:esri_boundaries>,
 <User username:esri_demographics>,
 <User username:esri_livingatlas>,
 <User username:esri_nav>,
 <User username:harnessuser1>,
 <User username:harnessuser2>,
 <User username:harnessuser3>,
 <User username:harnessuser4>,
 <User username:s_user1>,
 <User username:s_user2>,
 <User username:s_user3>,
 <User username:sharing1>,
 <User username:sharing2>,
 <User username:sharing3>,
 <User username:system_publisher>,
 <User username:test90>]

In [3]:
targetusers = target.users.search()
targetusers

[<User username:admin>,
 <User username:esri_boundaries>,
 <User username:esri_demographics>,
 <User username:esri_livingatlas>,
 <User username:esri_nav>,
 <User username:harnessuser1>,
 <User username:harnessuser2>,
 <User username:harnessuser3>,
 <User username:harnessuser4>,
 <User username:portaladmin>,
 <User username:robe5155>,
 <User username:sharing1>,
 <User username:sharing2>,
 <User username:sharing3>,
 <User username:system_publisher>]

If source users are already in the target, run the following code to delete them:

In [4]:
# create a list of Portal system users that shouldn't be messed with
systemusers = ['system_publisher', 'esri_nav', 'esri_livingatlas', 'esri_boundaries', 'esri_demographics']
for srcuser in sourceusers:
    if not srcuser.username in systemusers:
        try:
            targetusr = target.users.get(srcuser.username)
            if targetusr is not None:
                targetusr.reassign_to('portaladmin')
                targetusr.delete()
        except:
            print('User {} does not exist in Target Portal'.format(srcuser.username))

User automate1 does not exist in Target Portal
User s_user1 does not exist in Target Portal
User s_user2 does not exist in Target Portal
User s_user3 does not exist in Target Portal
User test90 does not exist in Target Portal


##Copy Users

In [5]:
def copy_user(target, user, password):
    # See if the user has firstName and lastName properties
    try:
        firstname = user.firstName
        lastname = user.lastName
    except:
        # if not, split the fullName
        fullName = user.fullName
        firstname = fullName.split()[0]
        try:
            lastname = fullName.split()[1]
        except:
            lastname = 'NoLastName'

    try:
        # create user; assume built-in users
        target_user = target.users.create(user.username, password, firstname, lastname,
                                          user.email, user.description)

        # update user properties
        target_user.update(user.access, user.preferredView,
                           user.description, user.tags, user.get_thumbnail_link(),
                           culture=user.culture, region=user.region)

        # update user role; assumes no custom roles
        if 'role' in user and not user.role == 'org_user':
            target_user.update_role(user.role)

        return target_user
    
    except:
        print("Unable to create user "+ user.username)
        return None

For each user in source portal, make a corresponding user in target portal:

In [6]:
for user in sourceusers:
    if not user.username in systemusers:
        print('Copying {}...'.format(user.username))
        copy_user(target, user, 'TestPassword@123')

Copying admin...
Copying automate1...
Copying harnessuser1...
Copying harnessuser2...
Copying harnessuser3...
Copying harnessuser4...
Copying s_user1...
Copying s_user2...
Copying s_user3...
Copying sharing1...
Copying sharing2...
Copying sharing3...
Copying test90...


Verify that users have been added to target portal:

In [7]:
targetusers = target.users.search()
targetusers

[<User username:admin>,
 <User username:automate1>,
 <User username:esri_boundaries>,
 <User username:esri_demographics>,
 <User username:esri_livingatlas>,
 <User username:esri_nav>,
 <User username:harnessuser1>,
 <User username:harnessuser2>,
 <User username:harnessuser3>,
 <User username:harnessuser4>,
 <User username:portaladmin>,
 <User username:robe5155>,
 <User username:sharing1>,
 <User username:sharing2>,
 <User username:sharing3>,
 <User username:s_user1>,
 <User username:s_user2>,
 <User username:s_user3>,
 <User username:system_publisher>,
 <User username:test90>]

#Groups

List the groups in the source and target portals:

In [8]:
sourcegroups = source.groups.search()
sourcegroups

[<Group title:"Esri Boundary Layers" owner:esri_boundaries>,
 <Group title:"Esri Demographic Layers" owner:esri_demographics>,
 <Group title:"Featured Maps and Apps" owner:admin>,
 <Group title:"Group1" owner:sharing1>,
 <Group title:"Living Atlas" owner:esri_livingatlas>,
 <Group title:"Living Atlas Analysis Layers" owner:esri_livingatlas>,
 <Group title:"Navigator Maps" owner:esri_nav>,
 <Group title:"so_grp_9_19_2016_14_39_12" owner:admin>,
 <Group title:"testgroMon Sep 19 2016 14:32:41 GMT-0700 (Pacific Daylight Time)" owner:admin>,
 <Group title:"testgroMon Sep 19 2016 15:30:37 GMT-0700 (Pacific Daylight Time)" owner:admin>,
 <Group title:"testgroMon Sep 19 2016 15:37:33 GMT-0700 (Pacific Daylight Time)" owner:admin>,
 <Group title:"testgroMon Sep 19 2016 15:41:45 GMT-0700 (Pacific Daylight Time)" owner:admin>]

In [9]:
targetgroups = target.groups.search()
targetgroups

[<Group title:"Esri Boundary Layers" owner:esri_boundaries>,
 <Group title:"Esri Demographic Layers" owner:esri_demographics>,
 <Group title:"Featured Maps and Apps" owner:portaladmin>,
 <Group title:"Living Atlas" owner:esri_livingatlas>,
 <Group title:"Living Atlas Analysis Layers" owner:esri_livingatlas>,
 <Group title:"Navigator Maps" owner:esri_nav>]

If source groups are already in the target, run the following code to delete them:

In [10]:
for tg in targetgroups:
    for sg in sourcegroups:
        if sg.title == tg.title and (not tg.owner in systemusers):
            print("Cleaning up group {} in target Portal...".format(tg.title))
            tg.delete()
            break

Cleaning up group Featured Maps and Apps in target Portal...


##Copy Groups

In [11]:
import tempfile

GROUP_COPY_PROPERTIES = ['title', 'description', 'tags', 'snippet', 'phone', 'access', 'isInvitationOnly']

def copy_group(target, source, group):
    """ Copy group to the target portal."""
    with tempfile.TemporaryDirectory() as temp_dir:
        # Create a new groups with the subset of properties we want to
        # copy to the target portal. Handle switching between org and
        # public access when going from an org in a multitenant portal
        # and a single tenant portal
        target_group = {}
        
        for property_name in GROUP_COPY_PROPERTIES:
            target_group[property_name] = group[property_name]

        if target_group['access'] == 'org' and target.properties['portalMode'] == 'singletenant':
            target_group['access'] = 'public'
        elif target_group['access'] == 'public'\
             and source.properties['portalMode'] == 'singletenant'\
             and target.properties['portalMode'] == 'multitenant'\
             and 'id' in target.properties: # is org
            target_group['access'] = 'org'

        # Handle the thumbnail (if one exists)
        thumbnail_file = None
        if 'thumbnail' in group:
            target_group['thumbnail'] = group.download_thumbnail(temp_dir)

        # Create the group in the target portal
        copied_group = target.groups.create_from_dict(target_group)
        
         # Reassign all groups to correct owners, add users, and find shared items
        members = group.get_members()
        copied_group.reassign_to(members['owner'])
        if members['users']:
            copied_group.add_users(members['users'])
        return copied_group

For each group in source portal, make a sorresponding group in target portal:

In [13]:
copied_groups = {}
for group in sourcegroups:
    if not group.owner in systemusers:
        tgt_group = copy_group(target, source, group)
        copied_groups[group.groupid] = tgt_group.groupid

Verify that groups have been created in the target portal:

In [15]:
targetgroups = target.groups.search()
targetgroups

[<Group title:"Esri Boundary Layers" owner:esri_boundaries>,
 <Group title:"Esri Demographic Layers" owner:esri_demographics>,
 <Group title:"Featured Maps and Apps" owner:admin>,
 <Group title:"Group1" owner:sharing1>,
 <Group title:"Living Atlas" owner:esri_livingatlas>,
 <Group title:"Living Atlas Analysis Layers" owner:esri_livingatlas>,
 <Group title:"Navigator Maps" owner:esri_nav>,
 <Group title:"so_grp_9_19_2016_14_39_12" owner:admin>,
 <Group title:"testgroMon Sep 19 2016 14:32:41 GMT-0700 (Pacific Daylight Time)" owner:admin>,
 <Group title:"testgroMon Sep 19 2016 15:30:37 GMT-0700 (Pacific Daylight Time)" owner:admin>,
 <Group title:"testgroMon Sep 19 2016 15:37:33 GMT-0700 (Pacific Daylight Time)" owner:admin>,
 <Group title:"testgroMon Sep 19 2016 15:41:45 GMT-0700 (Pacific Daylight Time)" owner:admin>]

#Items

Deduce the groups that each copied item will belong to:

In [26]:
source_items_by_id = {}
for user in sourceusers:
    if not user.username in systemusers:  # ignore any 'system' Portal users
        print("Collecting item ids for {}...".format(user.username))
        usercontent = user.items()
        # Copy item ids from root folder first
        for item in usercontent:
            source_items_by_id[item.itemid] = item 
        # Copy item ids from folders next
        folders = user.folders
        for folder in folders:
            folderitems = user.items(folder=folder['title'])
            for item in folderitems:
                source_items_by_id[item.itemid] = item   

In [28]:
for group in sourcegroups:
    if not group.owner in systemusers:
        target_group_id = copied_groups[group.groupid]
        for group_item in group.content():
            if not group_item.owner in systemusers:
                try:
                    item = source_items_by_id[group_item.itemid]
                    if item is not None:
                        if not 'groups'in item:
                            item['groups'] = []
                        item['groups'].append(target_group_id)
                except:
                    print("Not found item : " + group_item.itemid)

Not found item : 5adfee5ebfb24449bdd85d12dc61c5dd
Not found item : a2fd4796050f41c0a99144f39c39ef28


In [29]:
for key in source_items_by_id.keys():
    item = source_items_by_id[key]
    print("******\n"+item.title)
    if 'groups' in item:
        print(item.access)
        print(item.groups)

******
TestShareAsWFL_Folder
******
Pro_at030_RelativePath
******
HydrantInspections
******
cert-c6156-with a new name
******
TestLoadTilingSchemeFromFileSimpleJPEG
******
Pro_at024_Multipoint
******
AGO World Vehicle Routing Problem (portalproxy)
******
For_Del_protectionMon Sep 19 2016 15:42:24 GMT-0700 (Pacific Daylight Time)
******
cert-c6156-with a new name
******
AGO World Closest Facility Async (portalproxy)
******
url item 9_19_2016_15_38_27_10
******
foo


TypeError: Can't convert 'NoneType' object to str implicitly

##Copy Items

In [36]:
TEXT_BASED_ITEM_TYPES = frozenset(['Web Map', 'Feature Service', 'Map Service','Web Scene',
                                   'Image Service', 'Feature Collection', 'Feature Collection Template',
                                   'Web Mapping Application', 'Mobile Application', 'Symbol Set', 'Color Set',
                                   'Windows Viewer Configuration'])
ITEM_COPY_PROPERTIES = ['title', 'type', 'typeKeywords', 'description', 'tags',
                        'snippet', 'extent', 'spatialReference', 'name',
                        'accessInformation', 'licenseInfo', 'culture', 'url', ]

def copy_item(target, owner, folder, item):
    with tempfile.TemporaryDirectory() as temp_dir:
        copy_item = {}
        for property_name in ITEM_COPY_PROPERTIES:
            copy_item[property_name] = item[property_name]

        data_file = None
        if item.type in TEXT_BASED_ITEM_TYPES:
            # If its a text-based item, then read the text and add it to the request.
            if item.size > 0:
                text = item.get_data(False)
                #textstr = text.decode('utf-8')
                copy_item['text'] = text
        elif item.size > 0: # download data for all other types, not just item.type in FILE_BASED_ITEM_TYPES:
            # download data and add to the request as a file
            data_file = item.download(temp_dir)

        thumbnail_file = item.download_thumbnail(temp_dir)

        metadata_file = item.download_metadata(temp_dir)

        # Add the item to the target portal
        copied_item = target.content.add(copy_item, data_file, thumbnail_file, metadata_file, owner, folder)

        return copied_item

In [37]:
RELATIONSHIP_TYPES = frozenset(['Map2Service', 'WMA2Code',
                                'Map2FeatureCollection', 'MobileApp2Code', 'Service2Data',
                                'Service2Service'])

def copy_relationships(target, copied_items, src_item, relationships, owner, folder):
    
    target_item_id = copied_items.get(src_item.itemid)
    if target_item_id is not None:
        target_item = target.content.get(target_item_id)

        for rel_type in RELATIONSHIP_TYPES:
            src_rel_items = src_item.related_items(rel_type)

            for src_rel_item in src_rel_items:
                print("***Found related items for " + src_rel_item.title)
                source_rel_id = src_rel_item.itemid

                # See if it's already been copied to the target
                target_rel_id = copied_items.get(source_rel_id)
                if not target_rel_id:
                    # If not, then copy it to the target - folder may have moved though?
                    target_rel_item = clone_item(target, owner, folder, src_rel_item)

                    if target_rel_item is not None:
                        # add relationship from target_item to copied item
                        result = target_item.add_relationship(target_rel_item, rel_type)

                        if not result:
                            print('Unable to add relationship from ' +  target_item.itemid + ' to ' + target_rel_item.itemid)
                    else:
                        print("@@@Error Cloning Item "+src_rel_item.title)

In [ ]:
copied_items = {}
relationships = RELATIONSHIP_TYPES

for user in sourceusers:
    if not user.username in systemusers:
        print("**************\n"+user.username)
        usercontent = user.items()
        folders = user.folders
        for item in usercontent:
            copied_item = copy_item(target, user, None, item)
            if copied_item is not None:
                copied_items[item.itemid] = copied_item.itemid
                # share the item
                copied_item.share(item.access == 'public',
                                    item.access in ['org', 'public'],
                                    source_items_by_id[item.itemid].groups 
                                        if 'groups' in source_items_by_id[item.itemid]
                                        else None)
            else:
                print('Error copying ' + item.title)

        for folder in folders:
            target.content.create_folder(user, folder)
            folderitems = user.items(folder.title)
            for item in folderitems:
                #display(item)
                #print(item.__repr__())
                copied_item = copy_item(target, user, folder.title, item)
                if copied_item is not None:
                    copied_items[item.itemid] = copied_item.itemid

                    # share the item
                    copied_item.share(item.access == 'public',
                                      item.access in ['org', 'public'],
                                      source_items_by_id[item.itemid].groups 
                                          if 'groups' in source_items_by_id[item.itemid]
                                          else None)
                else:
                    print('Error copying ' + item.title)

        # Copy the related items for this user (if specified)
        if relationships:
            for folder in folders:
                folderitems = usercontent[folder]
                for item in folderitems:
                    copy_relationships(target, copied_items, item, relationships, user, folder)

**************
admin
